# Início rápido — AutoHybTS

Previsão de uma série univariada: a AutoHybTS compara modelos individuais e híbridos residuais (modelo-base + corretor do erro) e só aceita o híbrido quando ele comprova ganho na validação temporal. Caso contrário, fica com o modelo mais simples.

- Entrada: `pandas.Series` com índice temporal regular.
- `fh` é o horizonte de previsão (número de passos à frente).

In [1]:
import autohybts

print("AutoHybTS", autohybts.__version__)

AutoHybTS 0.0.1


## 1. Carregar dados

In [2]:
from autohybts.datasets import get_data

y = get_data("synthetic_daily")
y.tail()

2025-12-26    69.786655
2025-12-27    68.674617
2025-12-28    70.105403
2025-12-29    71.148468
2025-12-30    72.611886
Freq: D, Name: value, dtype: float64

## 2. Ajustar o experimento

`fit` valida a série (ordem, duplicatas, frequência), separa os últimos `fh` pontos como **teste selado** e congela as origens de validação. Nada do que vem depois enxerga o teste selado, até a avaliação final da seção 5.

In [3]:
from autohybts import AutoHybridExperiment

exp = AutoHybridExperiment(fh=14, session_id=42).fit(y)
exp.splits

,layer,train_start,origin,first_target,last_target,n_train,horizon
fold_id,,,,,,,
search-0,search,2024-01-01,2025-09-23,2025-09-24,2025-10-07,632,14
search-1,search,2024-01-01,2025-10-07,2025-10-08,2025-10-21,646,14
search-2,search,2024-01-01,2025-10-21,2025-10-22,2025-11-04,660,14
select-0,select,2024-01-01,2025-11-04,2025-11-05,2025-11-18,674,14
select-1,select,2024-01-01,2025-11-18,2025-11-19,2025-12-02,688,14
select-2,select,2024-01-01,2025-12-02,2025-12-03,2025-12-16,702,14


## 3. Comparar modelos

In [4]:
result = exp.compare_models()
result.leaderboard

,name,model,family,strategy,base,corrector,residual_source,window,alpha,mean_rank,...,MAE,RMSE,sMAPE,coverage,n_units,n_failed,n_skipped,fit_seconds,predict_seconds,is_champion
candidate_id,,,,,,,,,,,,,,,,,,,,,
7dd1082a4ee4fe9df20683376c5688b4a23322240929c4c44a8140e266a10ae2,mlp,mlp(lags=7),neural,standalone,mlp,NaN,NaN,NaN,NaN,1.0,...,1.329098,1.590181,0.019224,1.0,3,0,0,0.837549,0.011928,False
9fd962cca51322c0e3697027ac96d1fa8d35c2672647b0030737ef663dde6df6,mlp-linear,"mlp(lags=7) + linear [w=7, alpha=0.5, historic...",residual,residual_additive,mlp,linear,historical_oof_1step,7.0,0.5,2.0,...,1.335050,1.622713,0.019335,1.0,3,0,0,2.537739,0.036670,False
70ad174e1cc49eb256646399fb3e4b7d9df82f14f2978d09d35dac0019c7ef37,knn,knn(lags=7),kernel,standalone,knn,NaN,NaN,NaN,NaN,3.0,...,1.563061,1.966950,0.022621,1.0,3,0,0,0.142267,0.050971,False
d90528888825a4459816afbcf249ff909747924f0a2bbaa6e09d027e31342238,seasonal_naive,seasonal_naive(season_length=7),statistical,standalone,seasonal_naive,NaN,NaN,NaN,NaN,4.0,...,1.614291,1.909265,0.023346,1.0,3,0,0,0.000106,0.000261,False
1f8d8f124a8728e54358f736b6cef1744bd8df58ebf5b13403a2dd2723163f06,random_forest,random_forest(lags=7),tree,standalone,random_forest,NaN,NaN,NaN,NaN,5.0,...,1.634547,1.953615,0.023618,1.0,3,0,0,1.695656,0.563316,True
e9e747a9648ad8d377f20122dce611a43dd3bf85a64c950a8e1ef246ebc032b4,linear,linear(lags=7),linear,standalone,linear,NaN,NaN,NaN,NaN,6.0,...,1.649719,1.852607,0.023763,1.0,3,0,0,2.461917,0.015735,False
773d3b8c8d625571d4d4b4f4aa234d9f88b9b7fc0ceb3101c126e49e100378fc,seasonal_naive-linear,"seasonal_naive(season_length=7) + linear [w=7,...",residual,residual_additive,seasonal_naive,linear,historical_oof_1step,7.0,0.5,7.0,...,1.673298,1.998234,0.024205,1.0,3,0,0,0.284481,0.008640,False
c4821c0a39e39a26ca17be31d8298f1ebcaed90d98e0a1fd3722f7c6ed592f9b,linear-linear,"linear(lags=7) + linear [w=7, alpha=0.5, histo...",residual,residual_additive,linear,linear,historical_oof_1step,7.0,0.5,8.0,...,1.698826,1.914754,0.024472,1.0,3,0,0,0.612864,0.018723,False
972a9d57c62d5265bf431a410bf8ccd45ae226f2e6bb124bc9d763258cbdbae4,mlp-linear,"mlp(lags=7) + linear [w=7, alpha=1.0, historic...",residual,residual_additive,mlp,linear,historical_oof_1step,7.0,1.0,9.0,...,1.725096,2.009527,0.024833,1.0,3,0,0,1.738890,0.018608,False


In [5]:
best = result.best
best

CandidateSpec(family='tree', strategy='standalone', base=ModelSpec(name='random_forest', params_json='{"lags":7}'), corrector=None, window_size=None, alpha=None, residual_source=None, variant='A', members=(), options_json='{}', candidate_id='1f8d8f124a8728e54358f736b6cef1744bd8df58ebf5b13403a2dd2723163f06')

## 4. Prever

`finalize_model` reajusta a estrutura escolhida no histórico anterior ao teste, sem reabrir a seleção. `predict_model` prevê `fh` passos a partir do fim desse histórico.

In [6]:
final = exp.finalize_model(best)
forecast = exp.predict_model(final)
forecast.predictions

,series_id,origin,horizon,y_pred
timestamp,,,,
2025-12-17,__single__,2025-12-16,1,73.378733
2025-12-18,__single__,2025-12-16,2,71.898770
2025-12-19,__single__,2025-12-16,3,70.122112
2025-12-20,__single__,2025-12-16,4,68.294473
2025-12-21,__single__,2025-12-16,5,68.671440
2025-12-22,__single__,2025-12-16,6,70.818109
2025-12-23,__single__,2025-12-16,7,73.083289
2025-12-24,__single__,2025-12-16,8,73.790339
2025-12-25,__single__,2025-12-16,9,72.286542


## 5. Avaliar no teste selado

Feita uma única vez, depois que o modelo está congelado. O resultado não volta para a busca.

In [7]:
external = exp.evaluate_external(final)
external.metrics

,MASE,MAE,RMSE,sMAPE
model,,,,
random_forest(lags=7),0.704631,1.103425,1.273037,0.015608


## Próximos passos

- `02_hybrid_selection.ipynb`: como a busca monta os híbridos e por que o campeão foi escolhido.
- `03_finalize_and_persist.ipynb`: salvar, carregar e prever sem teste selado.
- `04_rolling_one_step.ipynb`: variante B, previsão de um passo com observações chegando.
- `05_benchmark_m4.ipynb`: uma série do M4 com o teste oficial selado.